In [ ]:
# ==========================================================
# TAX TOOL - VERSION NPT (NGƯỜI PHỤ THUỘC) 1.4
# Cập nhật:
# - Trả lại thuật toán rút trích Số (\D) siêu bền của phiên bản cũ
# - Vá lỗi ngắt trang (Page Break) trên chuỗi số gốc
# ==========================================================

!pip install pymupdf openpyxl pandas -q

import re
import fitz
import pandas as pd
from google.colab import files
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

# ==========================================
# 1. UPLOAD FILE
# ==========================================
print("UPLOAD:")
print("- 1 file Tong hop NPT.xlsx")
print("- Toàn bộ file PDF (Mẫu 14, Mẫu 40) từ Cục Thuế\n")

uploaded = files.upload()

excel_file = None
pdf_files = []

for file in uploaded:
    if file.lower().endswith((".xlsx", ".xls")):
        excel_file = file
    elif file.lower().endswith(".pdf"):
        pdf_files.append(file)

if excel_file is None:
    raise Exception("Không tìm thấy file Excel.")

# ==========================================
# 2. ĐỌC PDF & PHÂN LOẠI MẪU 14 / MẪU 40
# ==========================================
print("\nĐANG PHÂN TÍCH PDF...")

pdf_14_texts = {}
pdf_40_results = {}

for pdf in pdf_files:
    text = ""
    doc = fitz.open(pdf)
    for page in doc:
        # BỎ sort=True, đọc theo luồng gốc của PDF để không làm nát bảng biểu
        text += page.get_text()

    # Nếu là Mẫu 40 (Trùng MST - Không thành công)
    if "40/TB" in text or "Mẫu số: 40" in text:
        text_clean = re.sub(r'\s+', ' ', text)

        # Regex chống trôi dòng
        pattern = r'\b(\d{12})\b(?:(?!\b\d{12}\b).)*?MST(?:(?!\b\d{12}\b).)*?(\b\d{10}\b)'
        matches = re.findall(pattern, text_clean, flags=re.IGNORECASE | re.DOTALL)

        for cccd, mst in matches:
            pdf_40_results[cccd] = {
                "mst": mst,
                "status": "TRUNG_MST",
                "note": f"MST đã tồn tại: {mst}",
                "source": pdf
            }
        print(f"{pdf:<30} -> {len(matches)} hồ sơ trùng MST")

    # Nếu là Mẫu 14 (Cấp mới - Thành công)
    elif "14-MST" in text or "14- MST" in text or "Mẫu số: 14" in text:
        # TRẢ LẠI THUẬT TOÁN CŨ CỦA BẠN: Xóa sạch mọi thứ không phải là số
        text_only_digits = re.sub(r'\D', '', text)
        pdf_14_texts[pdf] = text_only_digits
        print(f"{pdf:<30} -> Đã nạp dữ liệu cấp mới")

    else:
        print(f"{pdf:<30} -> Không nhận diện được mẫu")

# ==========================================
# 3. ĐỌC VÀ CẬP NHẬT EXCEL
# ==========================================
print("\nĐANG CẬP NHẬT EXCEL...")
wb = load_workbook(excel_file)
ws = wb.active

last_col = ws.max_column
status_col = get_column_letter(last_col + 1)
note_col = get_column_letter(last_col + 2)
source_col = get_column_letter(last_col + 3)

ws[f"{status_col}3"] = "TRANG_THAI"
ws[f"{note_col}3"] = "GHI_CHU"
ws[f"{source_col}3"] = "FILE_NGUON"

green_fill = PatternFill(fill_type="solid", start_color="C6EFCE")
yellow_fill = PatternFill(fill_type="solid", start_color="FFF2CC")

updated = 0
duplicate = 0

for row in range(4, ws.max_row + 1):
    cccd_val = ws[f"K{row}"].value

    if cccd_val is None:
        continue

    # Làm sạch CCCD gốc từ Excel
    cccd = re.sub(r'\D', '', str(cccd_val).replace(".0", "")).zfill(12)
    if not cccd:
        continue

    found = False

    # BƯỚC 1: TÌM TRONG MẪU 14 (ƯU TIÊN CẤP MỚI)
    for pdf_name, text_only_digits in pdf_14_texts.items():
        # Dò bình thường (như code cũ của bạn)
        if cccd in text_only_digits:
            ws[f"D{row}"] = cccd
            ws[f"{status_col}{row}"] = "CAP_MOI"
            ws[f"{note_col}{row}"] = "Được cấp MST"
            ws[f"{source_col}{row}"] = pdf_name
            ws[f"D{row}"].fill = green_fill
            updated += 1
            found = True
            break
        else:
            # THUẬT TOÁN MỚI: Khớp nối khoảng hở trên chuỗi số
            # (Phòng trường hợp CCCD bị chèn số trang/ngày tháng vào giữa do ngắt trang)
            is_split_match = False

            for i in range(4, 9): # Cắt thử ở vị trí số 4 đến 8
                part1 = cccd[:i]
                part2 = cccd[i:]
                # Tìm part 1, bỏ qua tối đa 50 con số rác nằm giữa, rồi tìm part 2
                pattern = f"{part1}\\d{{1,50}}{part2}"

                if re.search(pattern, text_only_digits):
                    is_split_match = True
                    break

            if is_split_match:
                ws[f"D{row}"] = cccd
                ws[f"{status_col}{row}"] = "CAP_MOI"
                ws[f"{note_col}{row}"] = "Được cấp MST (Khớp 2 mảnh do ngắt trang)"
                ws[f"{source_col}{row}"] = pdf_name
                ws[f"D{row}"].fill = green_fill
                updated += 1
                found = True
                break

    # BƯỚC 2: NẾU KHÔNG THÀNH CÔNG -> TÌM MẪU 40
    if not found:
        if cccd in pdf_40_results:
            result = pdf_40_results[cccd]
            ws[f"D{row}"] = result["mst"]
            ws[f"{status_col}{row}"] = result["status"]
            ws[f"{note_col}{row}"] = result["note"]
            ws[f"{source_col}{row}"] = result["source"]
            ws[f"D{row}"].fill = yellow_fill
            duplicate += 1
            updated += 1
            found = True

# ==========================================
# 4. LƯU & TẢI XUỐNG
# ==========================================
output = "Tong_hop_NPT_Updated.xlsx"
wb.save(output)

print("\n==============================")
print("TỔNG KẾT HỒ SƠ NPT:")
print("==============================")
print(f"Tổng cập nhật  : {updated}")
print(f"- Cấp mới (12s): {updated - duplicate}")
print(f"- Trùng MST    : {duplicate}")
print(f"Không tìm thấy : {ws.max_row - 3 - updated}")
print("==============================")

files.download(output)